In [11]:
import cv2 as cv
import numpy as np
import json
from scipy.spatial.transform import Rotation
robot_file = "rob_coords.txt"


#cam_coords = np.array([[-211, -66, 931], [-135, -44, 981], [-226, -17, 1139]])
#robot_coords = np.array([[384.83, -129.80, -48.68], [443.55, -65.22, -50.70], [313.84, 62.25, -53.30]])




Get coordinates from Robot and Camera, must be the same amount or it will crash. Use the coordinates (in list format) with estimate_rigid_transform, and you can use rms_alignemt_error to calculate the root mean square error to see how good the calibration is (less is better). From the estimate_rigid_transform you get a rotation matrix and translation vector, to only get a homogeneous matrix pass both matrix and vector to build_homogeneous(). Then you can pass coordinates of a point in the camera frame that you want to convert to robot frame into convert_coordinates (XYZ) also send the homogeneous matrix along with it. You can also use the convert function to skip the middle step of calculation the homogeneous matrix, this however is less optimal as the homogenous matrix is "calculated" everytime the function is called. 

In [ ]:
# SHOUTOUT https://github.com/RealManRobot/hand_eye_calibration/blob/main/compute_to_hand.py
def convert(x ,y ,z, rotation_matrix, translation_vector): # X Y Z coordinates that should be translated into robot frame coordinates
    obj_camera_coordinates = np.array([x, y, z])
    T_camera_to_base_effector = np.eye(4)
    T_camera_to_base_effector[:3, :3] = rotation_matrix
    T_camera_to_base_effector[:3, 3] = translation_vector.reshape(3)

    # Compute the pose of the object against the base
    obj_camera_coordinates_homo = np.append(obj_camera_coordinates, [1])  # Convert object coordinates to homogeneous coordinates
    obj_base_effector_coordinates_homo = T_camera_to_base_effector.dot(obj_camera_coordinates_homo)
    obj_base_coordinates = obj_base_effector_coordinates_homo[:3]  

    #rot_matrix_homo = T_camera_to_base_effector[:3, :3]
    #quaternion = Rotation.from_matrix(rot_matrix_homo).as_quat()

    #return list(obj_base_coordinates), list(quaternion)
    return np.around(obj_base_coordinates,2).tolist()


def build_homogeneous(rotation_matrix, translation_vector):
    T_camera_to_base_effector = np.eye(4)
    T_camera_to_base_effector[:3, :3] = rotation_matrix
    T_camera_to_base_effector[:3, 3] = translation_vector.reshape(3)
    return T_camera_to_base_effector


def convert_coordinates(x ,y ,z, homogeneous_matrix): # X Y Z coordinates that should be translated into robot frame coordinates
    obj_camera_coordinates = np.array([x, y, z])
    obj_camera_coordinates_homo = np.append(obj_camera_coordinates, [1])  # Convert object coordinates to homogeneous coordinates
    obj_base_effector_coordinates_homo = homogeneous_matrix.dot(obj_camera_coordinates_homo)
    obj_base_coordinates = obj_base_effector_coordinates_homo[:3]  

    #return list(map(int, obj_base_coordinates)) # uncomment this line if you want to send integers instead of floats
    return np.around(obj_base_coordinates, 2).tolist() # Use this to get a list of new coordinates, chane the number to get the number of decimal numbers


def estimate_rigid_transform(camera_points, robot_points):
    cam = np.asarray(camera_points, dtype=np.float64)
    rob = np.asarray(robot_points,  dtype=np.float64)
    assert cam.shape == rob.shape and cam.shape[1] == 3 and cam.shape[0] >= 3, "Value error, differing amount of coordinates, Camera:" + str(len(cam)) + " Robot:"+ str(len(rob))
    

    camera_centroid = cam.mean(axis=0)
    robot_centroid  = rob.mean(axis=0)
    camera_centered = cam - camera_centroid
    robot_centered  = rob - robot_centroid

    cross_covariance = camera_centered.T @ robot_centered
    U, s, Vt = np.linalg.svd(cross_covariance)
    V = Vt.T

    # Ensure of proper rotation (det=+1)
    det_correction = np.sign(np.linalg.det(V @ U.T))
    rotation_matrix = V @ np.diag([1.0, 1.0, det_correction]) @ U.T

    translation_vector = robot_centroid - rotation_matrix @ camera_centroid
    return rotation_matrix, translation_vector

def rms_alignment_error(camera_points, robot_points, rotation_matrix, translation_vector):
    cam = np.asarray(camera_points, dtype=np.float64)
    rob = np.asarray(robot_points,  dtype=np.float64)
    t = np.asarray(translation_vector, dtype=np.float64).reshape(1, 3)
    predicted_robot_points = (rotation_matrix @ cam.T).T + t
    squared_errors = np.sum((predicted_robot_points - rob) ** 2, axis=1)
    rms_error = float(np.sqrt(np.mean(squared_errors)))
    return rms_error

def extract_data(file):
    float_list=[]
    with open(file, "r") as f:
        lines = f.readlines()
        for i in lines:
            x = json.loads(i)
            float_list.append(x)
    return list(float_list)


In [ ]:
#coords1 = np.random.random((3, 3))
#coords2 = np.random.random((3, 3))
#print(coords1, "\n", coords2)
#cam_coords = np.array([[-211, -66, 931], [-135, -44, 981], [-226, -17, 1139]])
#robot_coords = np.array([[384.83, -129.80, -48.68], [443.55, -65.22, -50.70], [313.84, 62.25, -53.30]])

cam_coords = extract_data("saved_coordinates.txt")
robot_coords = extract_data("rob_coords.txt")


In [15]:
#Rotation_matrix, translation_vector = estimate_rigid_transform(coords1, coords2)
#print(convert(10,10,10, Rotation_matrix, translation_vector))

print(cam_coords)
print(robot_coords)

[[-14.947012901306152, -55.55928421020508, 1232.8267822265625], [71.78984069824219, -67.73854064941406, 1184.242431640625], [-70.87337493896484, -48.76213836669922, 1278.7298583984375], [-132.69288635253906, -45.790618896484375, 1320.8858642578125], [-17.66465187072754, -7.576266288757324, 1456.97705078125], [71.45001220703125, -12.32824420928955, 1422.492431640625], [130.58103942871094, -22.190839767456055, 1422.492431640625], [202.30946350097656, -30.604692459106445, 1358.198974609375], [138.79368591308594, -41.88993453979492, 1271.964111328125], [47.52175521850586, -38.498958587646484, 1306.5283203125], [-80.89688873291016, -26.19223403930664, 1373.72119140625], [-68.02265930175781, -59.82250213623047, 1190.1051025390625], [-68.02265930175781, -57.75965881347656, 1190.1051025390625], [-9.913836479187012, -67.46484375, 1144.7677001953125], [36.62611389160156, -71.37825012207031, 1112.9686279296875]]
[[395, 20, -57], [519, 21, -75], [323, 24, -62], [254, 20, -46], [266, 231, -40], [38

In [16]:
# C_cam and B_robot are (N,3) arrays with matching point order
R, t = estimate_rigid_transform(cam_coords, robot_coords)
err = rms_alignment_error(cam_coords, robot_coords, R, t)
print("R =\n", R)
print("t =", t)
print("RMS error =", err)

R =
 [[ 0.92801826 -0.09578052 -0.36001139]
 [ 0.37229049  0.20345543  0.90554165]
 [-0.01348698 -0.974388    0.22446853]]
t = [  844.31307008 -1070.13208906  -386.03141046]
RMS error = 56.759297631718184


In [17]:
homogeneous = build_homogeneous(R,t)
print(convert_coordinates(100,100,100, homogeneous))

[891.54, -922.0, -462.37]


**MAIN LOOP(ish)**

In [20]:
import cv2 as cv
import numpy as np
from scipy.spatial.transform import Rotation


#cam_coords = np.array([[-211, -66, 931], [-135, -44, 981], [-226, -17, 1139]]) #SHOULD BE IN A LIST OF LISTS OR NUMPY ARRAY (it does work with some other types however)
#robot_coords = np.array([[384.83, -129.80, -48.68], [443.55, -65.22, -50.70], [313.84, 62.25, -53.30]]) #SHOULD BE IN A LIST OF LISTS OR NUMPY ARRAY (it does work with some other types however)

#### these should be done before a main loop
def build_homogeneous(rotation_matrix, translation_vector):
    T_camera_to_base_effector = np.eye(4)
    T_camera_to_base_effector[:3, :3] = rotation_matrix
    T_camera_to_base_effector[:3, 3] = translation_vector.reshape(3)
    return T_camera_to_base_effector


def convert_coordinates(x ,y ,z, homogeneous_matrix): # X Y Z coordinates that should be translated into robot frame coordinates
    obj_camera_coordinates = np.array([x, y, z])
    obj_camera_coordinates_homo = np.append(obj_camera_coordinates, [1])  # Convert object coordinates to homogeneous coordinates
    obj_base_effector_coordinates_homo = homogeneous_matrix.dot(obj_camera_coordinates_homo)
    obj_base_coordinates = obj_base_effector_coordinates_homo[:3]  

    #return list(map(int, obj_base_coordinates)) # uncomment this line if you want to send integers instead of floats
    return np.around(obj_base_coordinates,2).tolist() # Use this to get a list of new coordinates, chane the number to get the number of decimal numbers


def estimate_rigid_transform(camera_points, robot_points):
    cam = np.asarray(camera_points, dtype=np.float64)
    rob = np.asarray(robot_points,  dtype=np.float64)
    assert cam.shape == rob.shape and cam.shape[1] == 3 and cam.shape[0] >= 3, "Value error, differing amount of coordinates, Camera:" + str(len(cam)) + " Robot:"+ str(len(rob))
    

    camera_centroid = cam.mean(axis=0)
    robot_centroid  = rob.mean(axis=0)
    camera_centered = cam - camera_centroid
    robot_centered  = rob - robot_centroid

    cross_covariance = camera_centered.T @ robot_centered
    U, s, Vt = np.linalg.svd(cross_covariance)
    V = Vt.T

    # Ensure of proper rotation (det=+1)
    det_correction = np.sign(np.linalg.det(V @ U.T))
    rotation_matrix = V @ np.diag([1.0, 1.0, det_correction]) @ U.T

    translation_vector = robot_centroid - rotation_matrix @ camera_centroid
    return rotation_matrix, translation_vector

def rms_alignment_error(camera_points, robot_points, rotation_matrix, translation_vector):
    cam = np.asarray(camera_points, dtype=np.float64)
    rob = np.asarray(robot_points,  dtype=np.float64)
    t = np.asarray(translation_vector, dtype=np.float64).reshape(1, 3)
    predicted_robot_points = (rotation_matrix @ cam.T).T + t
    squared_errors = np.sum((predicted_robot_points - rob) ** 2, axis=1)
    rms_error = float(np.sqrt(np.mean(squared_errors)))
    return rms_error


rotation_matrix, translation_vector = estimate_rigid_transform(cam_coords, robot_coords)
err = rms_alignment_error(cam_coords, robot_coords, rotation_matrix, translation_vector)
print("R =\n", rotation_matrix)
print("t =", translation_vector)
print("RMS error =", err) #less than 1 is good if we are working in mm (which we are)
homogeneous = build_homogeneous(rotation_matrix,translation_vector)
###################################################################


print(convert_coordinates(150,-60,1212.678, homogeneous)) #coordinates x:100, y:100, z:100 in the robots frame

R =
 [[ 0.92801826 -0.09578052 -0.36001139]
 [ 0.37229049  0.20345543  0.90554165]
 [-0.01348698 -0.974388    0.22446853]]
t = [  844.31307008 -1070.13208906  -386.03141046]
RMS error = 56.759297631718184
[552.68, 71.63, -57.38]


[ 0.956132,  0.046079, -0.289289,  859.1997],
    [ 0.270961,  0.236151,  0.933174, -931.6338],
    [ 0.111315, -0.970623,  0.213306, -287.6907],
    [ 0.0,       0.0,       0.0,         1.0   ],